# Build USL-Suspilne Dataset

End-to-end pipeline: Firebase → annotations → download videos → split → crop signer → identify signers → extract poses.

**Output:** `data/usl-suspilne/` with `annotations.csv`, `{train,dev,test}.csv`, `features/{videoId}/{clip}.mp4`, `signers.json`, and `poses/mediapipe_holistic/{videoId}/{clip}.npy`

**Requirements:** `pip install firebase-admin yt-dlp num2words pymorphy3 pymorphy3-dicts-uk mediapipe opencv-python insightface onnxruntime scikit-learn`, `ffmpeg` installed, `serviceAccountKey.json` in repo root.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

# Dataset directory — final, publishable artefacts only (features + poses + split CSVs)
DATASET_DIR = ROOT / "data/usl-suspilne"
FEATURES_DIR = DATASET_DIR / "features"
HOLISTIC_DIR = DATASET_DIR / "poses/mediapipe_holistic"

# Cache/intermediate files — not part of the dataset
FIREBASE_DIR = ROOT / "data/firebase"
CACHE_DIR = ROOT / "data/cache"
ANNOTATIONS_CSV = CACHE_DIR / "annotations.csv"
SPLITS_CSV = CACHE_DIR / "splits.csv"
RAW_VIDEOS_DIR = CACHE_DIR / "raw_videos"
UNCROPPED_DIR = CACHE_DIR / "uncropped"
SIGNERS_JSON = CACHE_DIR / "signers.json"
SIGNER_PREVIEWS = CACHE_DIR / "signer_previews"

print(f"Project root: {ROOT}")

Project root: /Users/xandro/code/diploma/ucu-text-to-sign


## 1. Download Firebase Export

In [2]:
from download_firebase import download_and_save

export_path = download_and_save(output_dir=FIREBASE_DIR)
print(f"\nExport saved to: {export_path}")

Saved to /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-20.json
Updated /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/latest.json -> 2026-04-20.json
  Videos: 26
  Video captions entries: 26
  Total captions: 6801
  Aligned captions: 5159

Export saved to: /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-20.json


## 2. Build Annotations

Reads the Firebase export (data/firebase/latest.json), finds all captions marked as `aligned`, and writes:
- `data/usl-suspilne/annotations.csv` — `name|text|text_norm|annotator|signer_id` (for training)
- `data/cache/splits.csv` — `name|video|start|end` (for download/split)

`text_norm` is the normalized text: lowercased, punctuation stripped, numbers converted to words.
`signer_id` is filled from `data/usl-suspilne/signers.json` if it exists (see step 6). On the first run it will be blank; rerun this cell after step 6 to populate it.

Filters out clips shorter than 0.3s and any excluded video IDs.

In [3]:
from build_annotations_from_firebase import build_annotations

result = build_annotations(
    export_path=FIREBASE_DIR / "latest.json",
    output_csv=ANNOTATIONS_CSV,
    splits_csv=SPLITS_CSV,
    dataset_dir=DATASET_DIR,
    signers_path=SIGNERS_JSON,
)
print(f"\n{result['rows']} annotations from {len(result['videos'])} videos")

Videos: 25 complete, 1 incomplete
  train: 4344 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/train.csv
  dev: 418 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/dev.csv
  test: 263 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/test.csv
  test_unseen: 108 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/test_unseen.csv
Wrote 5133 annotations from 25 videos to /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/annotations.csv
Skipped 1 incomplete videos (use --all to include)
  0ULOz5HM4pA: 227 clips
  2Nnz697BVTw: 85 clips
  4FUDnWC9UJA: 308 clips
  6O0ZiSgKJNc: 245 clips
  82dy0zC6X_8: 108 clips
  9NMtlqDBY_s: 354 clips
  A2hCZVvtUSE: 122 clips
  IOflFDS2biE: 484 clips
  K9ouFMtz-s8: 57 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 116 clips
  Nyykyn4FpNo: 214 clips
  Q3yRVXmZdGQ: 142 clips
  S0o1oJ6G5qw: 115 clips
  SG9xYYOLBNI: 36 clips
  VVjY5HVY0jg: 347 clips
  cNT6ajjEwVU: 173 clip

## 3. Download Videos

Downloads YouTube videos at best available quality.
Skips already-downloaded videos. Use `--force VIDEO_ID` to re-download.

In [4]:
from download_videos import download_all

dl_result = download_all(splits_csv=SPLITS_CSV, dst=RAW_VIDEOS_DIR)

if dl_result["failed"]:
    print(f"\nFailed videos will be excluded in the verify step.")

Found 25 videos in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/splits.csv

[0ULOz5HM4pA]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/0ULOz5HM4pA.mp4 already exists
[2Nnz697BVTw]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/2Nnz697BVTw.mp4 already exists
[4FUDnWC9UJA]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/4FUDnWC9UJA.mp4 already exists
[6O0ZiSgKJNc]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/6O0ZiSgKJNc.mp4 already exists
[82dy0zC6X_8]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/82dy0zC6X_8.mp4 already exists
[9NMtlqDBY_s]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/9NMtlqDBY_s.mp4 already exists
[A2hCZVvtUSE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/A2hCZVvtUSE.mp4 already exists
[IOflFDS2biE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/ca

  [ERROR] yt-dlp failed for gtFOLKjnays:


## 4. Split into Clips

Splits videos into sentence-level clips based on timestamps.
Skips already-split clips. Use `--force VIDEO_ID` to re-split.

In [5]:
from split_videos import split_all

split_result = split_all(splits_csv=SPLITS_CSV, raw_dir=RAW_VIDEOS_DIR, dst=UNCROPPED_DIR)

Found 5133 annotations across 25 videos

[0ULOz5HM4pA] 227 clips
[2Nnz697BVTw] 85 clips
[4FUDnWC9UJA] 308 clips
[6O0ZiSgKJNc] 245 clips
[82dy0zC6X_8] 108 clips
[9NMtlqDBY_s] 354 clips
[A2hCZVvtUSE] 122 clips
[IOflFDS2biE] 484 clips
[K9ouFMtz-s8] 57 clips
[KUDt_SKkPUE] 146 clips
[MSqpwfErl34] 116 clips
[Nyykyn4FpNo] 214 clips
[Q3yRVXmZdGQ] 142 clips
[S0o1oJ6G5qw] 115 clips
[SG9xYYOLBNI] 36 clips
[VVjY5HVY0jg] 347 clips
[cNT6ajjEwVU] 173 clips
[d37lwXaSjs4] 500 clips
[eYEK-n2alOA] 180 clips
[g5Az2FHUQBU] 185 clips
[gtFOLKjnays] raw video not found, skipping 54 clips
[jj5jiyl2mh0] 423 clips
[uGMgleLkjho] 150 clips
[w_LdfLKP_0o] 71 clips
[yPYU48eSeBg] 291 clips

Done. 5079 clips, 0 failed, 1 videos skipped.
Skipped: gtFOLKjnays


## 5. Crop to Signer Region

Crops the bottom-right 510x510 region where the sign language interpreter appears.

In [6]:
from crop_signer import crop_all

crop_result = crop_all(src=UNCROPPED_DIR, dst=FEATURES_DIR)

Found 5083 clips in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/uncropped

Done. 5083 cropped, 0 failed.


## 6. Identify Signers (Face Clustering)

Runs RetinaFace + ArcFace on the cropped clips (via InsightFace) and clusters embeddings to assign a `signer_id` per clip. Videos with shift changes get per-clip ids; single-signer videos get one id shared by all their clips. Writes `data/usl-suspilne/signers.json` and a preview contact sheet under `data/usl-suspilne/signer_previews/`.

Re-run the **Build Annotations** cell above after this step to pick up the new `signer_id` column.

In [7]:
from identify_signers import identify_signers

signer_result = identify_signers(
    features_dir=FEATURES_DIR,
    output_path=SIGNERS_JSON,
    preview_dir=SIGNER_PREVIEWS,
)
print(f"\n{signer_result['n_clusters']} signers across {signer_result['videos']} videos")
if signer_result["multi_signer"]:
    print(f"Multi-signer videos: {signer_result['multi_signer']}")

Found 24 videos in /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/features

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xan

/opt/anaconda3/envs/usl/lib/python3.11/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  [OK]   0ULOz5HM4pA: 23/24 frames
  [OK]   2Nnz697BVTw: 24/24 frames
  [OK]   4FUDnWC9UJA: 24/24 frames
  [OK]   6O0ZiSgKJNc: 22/24 frames
  [OK]   82dy0zC6X_8: 24/24 frames
  [OK]   9NMtlqDBY_s: 21/24 frames
  [OK]   A2hCZVvtUSE: 23/24 frames
  [OK]   IOflFDS2biE: 23/24 frames
  [OK]   K9ouFMtz-s8: 24/24 frames
  [OK]   KUDt_SKkPUE: 24/24 frames
  [OK]   MSqpwfErl34: 24/24 frames
  [OK]   Nyykyn4FpNo: 23/24 frames
  [OK]   Q3yRVXmZdGQ: 24/24 frames
  [OK]   S0o1oJ6G5qw: 24/24 frames
  [OK]   SG9xYYOLBNI: 22/24 frames
  [OK]   VVjY5HVY0jg: 21/24 frames
  [OK]   cNT6ajjEwVU: 23/24 frames
  [OK]   d37lwXaSjs4: 24/24 frames
  [OK]   eYEK-n2alOA: 24/24 frames
  [OK]   g5Az2FHUQBU: 24/24 frames
  [OK]   jj5jiyl2mh0: 23/24 frames
  [OK]   uGMgleLkjho: 24/24 frames
  [OK]   w_LdfLKP_0o: 24/24 frames
  [OK]   yPYU48eSeBg: 24/24 frames

== Phase 1b: intra-video clustering to flag multi-signer videos ==
  [MULTI] 6O0ZiSgKJNc: sub-cluster sizes [15, 5, 2]
  [MULTI] IOflFDS2biE: sub-cluster sizes

/opt/anaconda3/envs/usl/lib/python3.11/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  6O0ZiSgKJNc: 244/245 clips embedded (1 without detections)
  IOflFDS2biE: 483/484 clips embedded (1 without detections)
  Nyykyn4FpNo: 212/214 clips embedded (2 without detections)
  d37lwXaSjs4: 499/500 clips embedded (1 without detections)

== Phase 3: global clustering ==
  samples=1458 → raw n_clusters=12 (threshold=0.4)
  after temporal smoothing: n_clusters=7

Done.
  multi-signer videos: ['6O0ZiSgKJNc', 'IOflFDS2biE', 'Nyykyn4FpNo', 'd37lwXaSjs4']
  mapping: /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/signers.json
  preview: /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/signer_previews/signers.png

7 signers across 24 videos
Multi-signer videos: ['6O0ZiSgKJNc', 'IOflFDS2biE', 'Nyykyn4FpNo', 'd37lwXaSjs4']


## 7. Extract Poses (MediaPipe Holistic)

Extracts 75 keypoints per frame: 33 body + 21 left hand + 21 right hand.
Output shape per clip: `(T, 225)` = 75 joints × (x, y, visibility).
Skips already-extracted clips.

In [8]:
from extract_poses import extract_all

extract_all(src=FEATURES_DIR, dst=HOLISTIC_DIR)

Found 5083 clips in /Users/xandro/code/diploma/ucu-text-to-sign/data/usl-suspilne/features

  [1/5083] 0ULOz5HM4pA/0000: 227 frames (cached)
  [2/5083] 0ULOz5HM4pA/0001: 74 frames (cached)
  [3/5083] 0ULOz5HM4pA/0002: 239 frames (cached)
  [4/5083] 0ULOz5HM4pA/0003: 136 frames (cached)
  [5/5083] 0ULOz5HM4pA/0004: 247 frames (cached)
  [6/5083] 0ULOz5HM4pA/0005: 164 frames (cached)
  [7/5083] 0ULOz5HM4pA/0006: 128 frames (cached)
  [8/5083] 0ULOz5HM4pA/0007: 194 frames (cached)
  [9/5083] 0ULOz5HM4pA/0008: 109 frames (cached)
  [10/5083] 0ULOz5HM4pA/0009: 268 frames (cached)
  [11/5083] 0ULOz5HM4pA/0010: 197 frames (cached)
  [12/5083] 0ULOz5HM4pA/0011: 293 frames (cached)
  [13/5083] 0ULOz5HM4pA/0012: 335 frames (cached)
  [14/5083] 0ULOz5HM4pA/0013: 127 frames (cached)
  [15/5083] 0ULOz5HM4pA/0014: 154 frames (cached)
  [16/5083] 0ULOz5HM4pA/0015: 49 frames (cached)
  [17/5083] 0ULOz5HM4pA/0016: 191 frames (cached)
  [18/5083] 0ULOz5HM4pA/0017: 34 frames (cached)
  [19/5083] 0ULOz5HM

I0000 00:00:1776686773.709494   53608 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776686773.759045   53612 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776686773.831065   53612 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776686773.844093   53623 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776686773.850746   53626 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776686773.870216   53632 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback t

  [219/5083] 0ULOz5HM4pA/0225: TIMEOUT (120s)
  [220/5083] 0ULOz5HM4pA/0226: 246 frames (cached)
  [221/5083] 0ULOz5HM4pA/0227: 19 frames (cached)
  [222/5083] 0ULOz5HM4pA/0228: 875 frames (cached)
  [223/5083] 0ULOz5HM4pA/0229: 253 frames (cached)
  [224/5083] 0ULOz5HM4pA/0230: 41 frames (cached)
  [225/5083] 0ULOz5HM4pA/0231: 162 frames (cached)
  [226/5083] 0ULOz5HM4pA/0232: 10 frames (cached)
  [227/5083] 0ULOz5HM4pA/0233: 173 frames (cached)
  [228/5083] 0ULOz5HM4pA/0234: 183 frames (cached)
  [229/5083] 2Nnz697BVTw/0000: 111 frames (cached)
  [230/5083] 2Nnz697BVTw/0001: 342 frames (cached)
  [231/5083] 2Nnz697BVTw/0002: 282 frames (cached)
  [232/5083] 2Nnz697BVTw/0003: 105 frames (cached)
  [233/5083] 2Nnz697BVTw/0004: 77 frames (cached)
  [234/5083] 2Nnz697BVTw/0005: 195 frames (cached)
  [235/5083] 2Nnz697BVTw/0006: 209 frames (cached)
  [236/5083] 2Nnz697BVTw/0007: 147 frames (cached)
  [237/5083] 2Nnz697BVTw/0008: 226 frames (cached)
  [238/5083] 2Nnz697BVTw/0009: 335 frame

I0000 00:00:1776686894.556278   57255 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776686894.592531   57258 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776686894.606565   57261 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776686894.609396   57267 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776686894.612217   57269 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776686894.614883   57269 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  [2758/5083] SG9xYYOLBNI/0033: TIMEOUT (120s)
  [2759/5083] SG9xYYOLBNI/0034: 111 frames (cached)
  [2760/5083] SG9xYYOLBNI/0035: 130 frames (cached)
  [2761/5083] SG9xYYOLBNI/0036: 300 frames (cached)


I0000 00:00:1776687014.577261   60134 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776687014.612680   60136 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776687014.626540   60139 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776687014.629189   60146 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776687014.631993   60148 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776687014.634147   60155 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  [2762/5083] SG9xYYOLBNI/0037: TIMEOUT (120s)
  [2763/5083] VVjY5HVY0jg/0000: 24 frames (cached)
  [2764/5083] VVjY5HVY0jg/0001: 50 frames (cached)
  [2765/5083] VVjY5HVY0jg/0002: 47 frames (cached)
  [2766/5083] VVjY5HVY0jg/0003: 63 frames (cached)
  [2767/5083] VVjY5HVY0jg/0004: 69 frames (cached)
  [2768/5083] VVjY5HVY0jg/0005: 127 frames (cached)
  [2769/5083] VVjY5HVY0jg/0006: 113 frames (cached)
  [2770/5083] VVjY5HVY0jg/0007: 143 frames (cached)
  [2771/5083] VVjY5HVY0jg/0008: 49 frames (cached)
  [2772/5083] VVjY5HVY0jg/0009: 203 frames (cached)
  [2773/5083] VVjY5HVY0jg/0010: 128 frames (cached)
  [2774/5083] VVjY5HVY0jg/0011: 95 frames (cached)
  [2775/5083] VVjY5HVY0jg/0012: 44 frames (cached)
  [2776/5083] VVjY5HVY0jg/0013: 77 frames (cached)
  [2777/5083] VVjY5HVY0jg/0014: 107 frames (cached)
  [2778/5083] VVjY5HVY0jg/0015: 116 frames (cached)
  [2779/5083] VVjY5HVY0jg/0016: 92 frames (cached)
  [2780/5083] VVjY5HVY0jg/0017: 91 frames (cached)
  [2781/5083] VVjY5HVY0jg/00

I0000 00:00:1776687134.737519   63046 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776687134.772820   63048 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776687134.787782   63049 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776687134.790922   63057 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1776687134.793502   63058 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776687134.796452   63058 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  [3159/5083] cNT6ajjEwVU/0049: TIMEOUT (120s)
  [3160/5083] cNT6ajjEwVU/0050: 307 frames (cached)
  [3161/5083] cNT6ajjEwVU/0051: 280 frames (cached)
  [3162/5083] cNT6ajjEwVU/0052: 372 frames (cached)
  [3163/5083] cNT6ajjEwVU/0053: 178 frames (cached)
  [3164/5083] cNT6ajjEwVU/0054: 219 frames (cached)
  [3165/5083] cNT6ajjEwVU/0055: 291 frames (cached)
  [3166/5083] cNT6ajjEwVU/0056: 206 frames (cached)
  [3167/5083] cNT6ajjEwVU/0057: 144 frames (cached)
  [3168/5083] cNT6ajjEwVU/0058: 86 frames (cached)
  [3169/5083] cNT6ajjEwVU/0059: 27 frames (cached)
  [3170/5083] cNT6ajjEwVU/0060: 139 frames (cached)
  [3171/5083] cNT6ajjEwVU/0061: 231 frames (cached)
  [3172/5083] cNT6ajjEwVU/0062: 653 frames (cached)
  [3173/5083] cNT6ajjEwVU/0063: 435 frames (cached)
  [3174/5083] cNT6ajjEwVU/0064: 171 frames (cached)
  [3175/5083] cNT6ajjEwVU/0065: 110 frames (cached)
  [3176/5083] cNT6ajjEwVU/0066: 202 frames (cached)
  [3177/5083] cNT6ajjEwVU/0067: 188 frames (cached)
  [3178/5083] cNT6a

{'total': 5079, 'failed': 4}

import csv
import subprocess
import json

# Count total clips on disk
total_clips = len(list(FEATURES_DIR.glob("*/*.mp4")))
print(f"Total clips: {total_clips}")
for vid_dir in sorted(FEATURES_DIR.iterdir()):
    if vid_dir.is_dir():
        clips = list(vid_dir.glob("*.mp4"))
        print(f"  {vid_dir.name}: {len(clips)} clips")

def _filter_csv(path):
    """Drop rows whose clip file doesn't exist on disk; rewrite in place."""
    with open(path) as f:
        rows = list(csv.DictReader(f, delimiter="|"))
    valid = []
    missing = []
    for row in rows:
        vid, clip = row["name"].split("/")
        if (FEATURES_DIR / vid / f"{clip}.mp4").exists():
            valid.append(row)
        else:
            missing.append(row["name"])
    if missing:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(valid[0].keys()), delimiter="|")
            writer.writeheader()
            writer.writerows(valid)
    return len(valid), missing

# Filter all dataset CSVs (plus the cache copy of annotations.csv for consistency)
for path in [ANNOTATIONS_CSV,
             DATASET_DIR / "train.csv",
             DATASET_DIR / "dev.csv",
             DATASET_DIR / "test.csv"]:
    n_valid, missing = _filter_csv(path)
    status = f"{n_valid} rows"
    if missing:
        by_vid = {}
        for name in missing:
            v = name.split("/")[0]
            by_vid[v] = by_vid.get(v, 0) + 1
        status += f" (removed {len(missing)}: {by_vid})"
    print(f"  {path.name}: {status}")

In [9]:
import csv
import subprocess
import json

dataset_dir = ROOT / "data/usl-suspilne"

# Count annotations
with open(ANNOTATIONS_CSV) as f:
    annotations = list(csv.DictReader(f, delimiter="|"))
print(f"Annotations: {len(annotations)}")

# Count clips per video
for vid_dir in sorted(FEATURES_DIR.iterdir()):
    if vid_dir.is_dir():
        clips = list(vid_dir.glob("*.mp4"))
        print(f"  {vid_dir.name}: {len(clips)} clips")

total_clips = len(list(FEATURES_DIR.glob("*/*.mp4")))
print(f"\nTotal clips: {total_clips}")

# Check for missing clips and filter annotations
missing = []
valid = []
for row in annotations:
    clip = FEATURES_DIR / row["name"].split("/")[0] / (row["name"].split("/")[1] + ".mp4")
    if clip.exists():
        valid.append(row)
    else:
        missing.append(row["name"])

if missing:
    print(f"\nWARNING: {len(missing)} annotations without clips — removing from annotations.csv")
    missing_vids = {}
    for name in missing:
        vid = name.split("/")[0]
        missing_vids[vid] = missing_vids.get(vid, 0) + 1
    for vid, count in sorted(missing_vids.items()):
        print(f"  {vid}: {count} missing")

    fieldnames = list(valid[0].keys())
    with open(ANNOTATIONS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="|")
        writer.writeheader()
        writer.writerows(valid)
    print(f"\nFiltered annotations: {len(valid)} (removed {len(missing)})")
else:
    print("\nAll annotations have matching clips.")

Annotations: 5133
  0ULOz5HM4pA: 228 clips
  2Nnz697BVTw: 85 clips
  4FUDnWC9UJA: 308 clips
  6O0ZiSgKJNc: 245 clips
  82dy0zC6X_8: 108 clips
  9NMtlqDBY_s: 354 clips
  A2hCZVvtUSE: 122 clips
  IOflFDS2biE: 484 clips
  K9ouFMtz-s8: 57 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 116 clips
  Nyykyn4FpNo: 214 clips
  Q3yRVXmZdGQ: 142 clips
  S0o1oJ6G5qw: 115 clips
  SG9xYYOLBNI: 38 clips
  VVjY5HVY0jg: 347 clips
  cNT6ajjEwVU: 174 clips
  d37lwXaSjs4: 500 clips
  eYEK-n2alOA: 180 clips
  g5Az2FHUQBU: 185 clips
  jj5jiyl2mh0: 423 clips
  uGMgleLkjho: 150 clips
  w_LdfLKP_0o: 71 clips
  yPYU48eSeBg: 291 clips

Total clips: 5083

  gtFOLKjnays: 54 missing

Filtered annotations: 5079 (removed 54)


In [10]:
# Total duration
total_dur = 0
per_video = {}
for clip in sorted(FEATURES_DIR.glob("*/*.mp4")):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(clip)],
        capture_output=True, text=True
    )
    dur = float(json.loads(result.stdout)["format"]["duration"])
    total_dur += dur
    vid = clip.parent.name
    per_video[vid] = per_video.get(vid, 0) + dur

for vid in sorted(per_video):
    m, s = divmod(per_video[vid], 60)
    print(f"  {vid}: {int(m)}m {s:.0f}s")

h, remainder = divmod(total_dur, 3600)
m, s = divmod(remainder, 60)
print(f"\nTotal duration: {int(h)}h {int(m)}m {s:.0f}s")

  0ULOz5HM4pA: 29m 55s
  2Nnz697BVTw: 12m 50s
  4FUDnWC9UJA: 25m 20s
  6O0ZiSgKJNc: 22m 12s
  82dy0zC6X_8: 11m 56s
  9NMtlqDBY_s: 27m 11s
  A2hCZVvtUSE: 15m 32s
  IOflFDS2biE: 34m 28s
  K9ouFMtz-s8: 7m 17s
  KUDt_SKkPUE: 16m 27s
  MSqpwfErl34: 10m 38s
  Nyykyn4FpNo: 19m 17s
  Q3yRVXmZdGQ: 11m 22s
  S0o1oJ6G5qw: 9m 1s
  SG9xYYOLBNI: 20m 45s
  VVjY5HVY0jg: 26m 44s
  cNT6ajjEwVU: 26m 15s
  d37lwXaSjs4: 41m 24s
  eYEK-n2alOA: 16m 1s
  g5Az2FHUQBU: 14m 60s
  jj5jiyl2mh0: 37m 6s
  uGMgleLkjho: 13m 16s
  w_LdfLKP_0o: 9m 42s
  yPYU48eSeBg: 27m 33s

Total duration: 8h 7m 12s
